# Fine tune Llama Model

## Installing

Install all required library

In [ ]:
!pip install \
accelerate \
peft \
bitsandbytes \
transformers \
trl \
triton
!pip install --upgrade transformers accelerate

## Training

Import all required libraries

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import(
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging
)
from peft import  LoraConfig, PeftModel
from trl import SFTTrainer

In case of Llama 2, the following prompt templae is used for the chat models.

System Prompt(optional) to guide the model
User prompt(required) to give the instruction
Model Answer (required)




In [ ]:
# <s> [INST] <<SYS>>
# System prompt
# <</SYS>>

# User Prompt [/INS] Model answer </s>

We will reformat our instruction dataset fo follow Llama 2 template

* Original dataset: https://huggingface.co/datasets/timdettmers/openassistant-guanaco



How to fine tune Llama 2

* Free Google Colab offers a 15GB Graphics Card (Limited Resource -> Barely enough to store Llama 2-7b's weight).

* We also need to consider the overhead due to optimizer states, gradient, and forward action activations.

* Full fine tuning is not possible here: we need parameter-efficient fine-tuning (PEFT) techquies like LoRA, QLoRA.

* To drastically reduce the VRAM usage, we must fine-tune the model in 4-bit precision, which is why we'll use LoRA and QLoRA here.

1. Load llama2-7b model
2. Train it on the mlabonne/guanaco-llama2-1k which produce oru fine tuned lamma-2-7b-chat-finetune

QLoRA will use a rank of 64 with scaling parameter of 16. We'll load the Llama 2 model directly in 4-bit precision using the NF4 type and train it for one epoch

In [ ]:
model_name = "NousResearch/Llama-2-7b-chat-hf"

# dataset_name = "mlabonne/guanaco-llama2-1k"
dataset_name = "Amod/mental_health_counseling_conversations"

model_fine_tune_name = "Llama-2-7b-chat-hf-finetune"

In [ ]:
## QLoRA parameters
lora_r = 8  # rank
lora_alpha = 16
lora_dropout = 0.1
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]
## bitsandbytes parameters
use_4bit = True # activation 4-bit precision model base model loading
bnb_4bit_compute_dtype = 'float16' # compute dtype for 4-bit base models
bnb_4bit_quant_type = 'nf4' # Quantization type (fp4 or nf4)
use_nested_quant = False # activate nested quantization for 4-bit base models

## TrainingArguments Parameter

output_dir = "./results"
num_train_epochs = 1
# Enable fp16/bp16 training
fp16 = False
bf16 = False

per_device_train_batch_size = 1

per_device_eval_batch_size = 1

gradient_accmulation_steps = 4

gradient_checkpointing = True

max_grad_norm = 0.3 # Maximum gradient normal (gradient clipping)

learning_rate = 2e-4 # Initial learning rate (AdamW optimizer)

weight_decay = 0.001

optim = "paged_adamw_32bit" # optimizer to use

lr_schedule_type = "cosine" # Learning rate schedule

max_steps = -1 # No of training steps (override train epcohs)

warmup_ratio = 0.03 # Ratio of steps for a linear warmup (from 0 to learning rate)

group_by_length = True # group sequences into batches with same length

save_steps = 100

logging_steps = 25

## STF Parameter

max_seq_length = None
packing = False # pack multiple short example in the same input sequence to increase efficency
device_map = {"": 0} # Load the entire model on the GPU 0

Load everything and start the fine-tuning process

1. First of all we want to load the dataset we defined. Here our dataset is already preprocessed but, usually this is where you would reformat the prompt, filter out bax text, combine multiple datasets, etc.

2. Then, we're configuring bitsandbytes for 4-bit quantization.

3. Next, we're loading the Llama 2 model in 4-bit precision on a GPU with the corresponding tokenizer.

4. Finally, we're loading configurations for QLoRA, regular training parameters, and passing everything to the STF traniner. The training can finally start.

In [ ]:
# Load the dataset (you can process it here)
dataset = load_dataset(dataset_name, split='train')

In [ ]:
dataset['Response'][:1]

["If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.Bad feelings are part of living. \xa0They are the motivation to remove ourselves from situations and relationships which do us more harm than good.Bad feelings do feel terrible. \xa0 Your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today."]

In [ ]:
dataset['Context'][:1]

["I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?"]

In [ ]:
# def format_dataset(data):
#   formated_data = {'text': f"###user: {data['Context']}\n###psychologist: {data['Response']}"}
#   return formated_data

In [ ]:
def format_dataset(data):
    return {'text': f"<s>[INST] {data['Context']} [/INST] {data['Response']} </s>"}

In [ ]:
dataset = dataset.map(format_dataset)

In [ ]:
dataset['text'][:3]

["<s>[INST] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone? [/INST] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrib

In [ ]:
compute_type = getattr(torch, bnb_4bit_compute_dtype)

In [ ]:
compute_type

torch.float16

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_type,
    bnb_4bit_use_double_quant=use_nested_quant
)

In [ ]:
# Check GPU compatibility with bfloat16
if compute_type == torch.float16 and use_4bit:
  major, _ = torch.cuda.get_device_capability()
  if major >= 8:
    print("=" * 80)
    print("Your GPU support bfloat16: accelerate training with bf16=True")
  else:
    print(f"Major is {major}")

Major is 7


In [ ]:
# Load base model

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map=device_map,
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
print(next(model.parameters()).dtype)

torch.float16


In [ ]:
# loaded_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config = bnb_config,
#     device_map=device_map
# )

In [ ]:
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code = True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right' # Fix weird overflow issue with fp16 training


In [ ]:
dataset

Dataset({
    features: ['Context', 'Response', 'text'],
    num_rows: 3512
})

In [ ]:
lengths = []
for idx,example in enumerate(dataset):
    tokens = tokenizer(example["text"])["input_ids"]
    lengths.append((idx,len(tokens)))


In [ ]:
max(lengths,key=lambda x:x[1]),min(lengths,key=lambda x:x[1])

((2624, 26816), (2625, 19))

In [ ]:
dataset['text'][2624]

'<s>[INST] I get so much anxiety, and I don’t know why. I feel like I can’t do anything by myself because I’m scared of the outcomes. [/INST] The other two post answers to your question are very good and I don\'t feel the need to repeat what has already been said quite well, but I will offer one other option I have been able to utilize quite successfully with those dealing with panic attacks. \xa0Chain analysis is a fantastic way for your to map out the situation starting with the prompting event, the chain of events ((links) that lead up to the behavior - in this case a panic attack, and then what the consequences were. \xa0See the illustration below:<img src=" </s>'

In [ ]:
len(dataset['text'][19])

650

In [ ]:
import re

In [ ]:
def clean_html(example):
    text = example['text']
    pattern = r"<.*?\/>|<.*"
    cleaned = re.sub(pattern, "", text, flags=re.DOTALL)

    return {'cleaned_text': cleaned}

dataset = dataset.map(clean_html)

In [ ]:
dataset

Dataset({
    features: ['Context', 'Response', 'text', 'cleaned_text'],
    num_rows: 3512
})

In [ ]:
dataset['cleaned_text'][2624]

''

In [ ]:
lengths = []
for idx,example in enumerate(dataset):
    tokens = tokenizer(example["cleaned_text"])["input_ids"]
    lengths.append((idx,len(tokens)))


In [ ]:
max(lengths,key=lambda x:x[1]),min(lengths,key=lambda x:x[1])

((0, 1), (0, 1))

In [ ]:
max(lengths,key=lambda x:x[1]),min(lengths,key=lambda x:x[1])

((0, 1), (0, 1))

In [ ]:
import numpy as np

values = [v for i,v in lengths]
arr = np.array(values)
arr

array([1, 1, 1, ..., 1, 1, 1])

In [ ]:
q1 = np.percentile(arr,25)
q3 = np.percentile(arr,75)

In [ ]:
iqr = q3 - q1
upper_bound = (1.5 * iqr) + q3
upper_bound

np.float64(1.0)

In [ ]:
def tokenize(example):
    return tokenizer(
        example["cleaned_text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

In [ ]:
tokenized_dataset

Dataset({
    features: ['Context', 'Response', 'text', 'cleaned_text', 'input_ids', 'attention_mask'],
    num_rows: 3512
})

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=target_modules,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
total_steps = (len(tokenized_dataset) // gradient_accmulation_steps) * num_train_epochs
warmup_steps_calculated = int(total_steps * warmup_ratio)

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accmulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_steps=warmup_steps_calculated,
    lr_scheduler_type=lr_schedule_type,
    report_to='tensorboard'
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    args=training_arguments,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
trainer.train()

{'loss': '1.957', 'grad_norm': '0.1836', 'learning_rate': '0.0001846', 'entropy': '2.703', 'num_tokens': '5.12e+04', 'mean_token_accuracy': '0.6673', 'epoch': '0.02847'}
{'loss': '0.00054', 'grad_norm': '0', 'learning_rate': '0.0001996', 'entropy': '0.001854', 'num_tokens': '1.024e+05', 'mean_token_accuracy': '1', 'epoch': '0.05695'}
{'loss': '0', 'grad_norm': '0', 'learning_rate': '0.0001984', 'entropy': '0', 'num_tokens': '1.536e+05', 'mean_token_accuracy': '1', 'epoch': '0.08542'}
{'loss': '0', 'grad_norm': '0', 'learning_rate': '0.0001964', 'entropy': '0', 'num_tokens': '2.048e+05', 'mean_token_accuracy': '1', 'epoch': '0.1139'}
{'loss': '0', 'grad_norm': '0', 'learning_rate': '0.0001935', 'entropy': '0', 'num_tokens': '2.56e+05', 'mean_token_accuracy': '1', 'epoch': '0.1424'}
{'loss': '0', 'grad_norm': '0', 'learning_rate': '0.0001899', 'entropy': '0', 'num_tokens': '3.072e+05', 'mean_token_accuracy': '1', 'epoch': '0.1708'}
{'loss': '0', 'grad_norm': '0', 'learning_rate': '0.0001

In [ ]:
trainer.model.save_pretrained(model_fine_tune_name)

Check the plots on tensorboard, as follows

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/runs

In [ ]:
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = 'I am feeling very insecure. How to handle it?'
pipe = pipeline(task="text-generation",model=model,tokenizer=tokenizer)
result = pipe(f"<s>[INST] {prompt} [/INST]")

In [ ]:
print(result[0]['generated_text'])

In [ ]:
model

In [ ]:
# del model
# del pipe
# del trainer
# import gc
# gc.collect()
# # gc.collect()

## Inference

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel, PeftConfig, AutoPeftModelForCausalLM
import torch

In [ ]:
# lora_model_path = "fine_tuned_weights\Llama-2-7b-chat-hf-finetune"

In [ ]:
import zipfile
import os

# Path to the zip file
zip_file_path = '/content/saved_model.zip'

# Directory where you want to extract the contents
extract_to = '/content/model_weights'

# Unzipping the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(f"Files extracted to: {os.path.abspath(extract_to)}")
lora_model_path = "/content/model_weights/kaggle/working/Llama-2-7b-chat-hf-finetune"

In [ ]:
from huggingface_hub import login

login(new_session=False)

In [ ]:
peft_config = PeftConfig.from_pretrained(lora_model_path)
peft_config

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    lora_model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

In [ ]:
prompt = "I have no motivation to study. I am very distracted and have no focus?"
pipe = pipeline(task="text-generation",model=model,tokenizer=tokenizer)
result = pipe(f"<s>[INST] {prompt} [/INST]")

In [ ]:
result

In [ ]:
text = result[0]['generated_text']

import re
cleaned_text = re.sub(r"<.*?>|\[.*?\]", "", text).strip()

print(cleaned_text)

In [ ]:
from huggingface_hub import login

login(new_session=False)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="my-finetuned-model", private=False)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model.save_pretrained("./my-finetuned-model")
tokenizer.save_pretrained("./my-finetuned-model")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model.push_to_hub("ArjunRavi/my-finetuned-model")
tokenizer.push_to_hub("ArjunRavi/my-finetuned-model")


In [ ]:
instr = f"<s>[INST] {prompt} [/INST]"

inputs = tokenizer(instr, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.2,
        max_new_tokens=128,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)


In [ ]:
# peft_config = PeftConfig.from_pretrained(lora_model_path)
# print(peft_config,peft_config.base_model_name_or_path)
# base_model = AutoModelForCausalLM.from_pretrained(
#     peft_config.base_model_name_or_path,
#     return_dict=True,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

In [ ]:
print(peft_config)

In [ ]:
peft_config.base_model_name_or_path

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

In [ ]:
model = PeftModel.from_pretrained(base_model, lora_model_path,is_trainable=False)

In [ ]:
# print(model.eval())

In [ ]:
prompt = "I have no motivation to study. I am very distracted and have no focus?"

In [ ]:
pipe = pipeline(task="text-generation",model=model,tokenizer=tokenizer)
result = pipe(f"<s>[INST] {prompt} [/INST]")

In [ ]:
result

In [ ]:
text = result[0]['generated_text']

import re
cleaned_text = re.sub(r"<.*?>|\[.*?\]", "", text).strip()

print(cleaned_text)


In [ ]:
# from torch import cuda

# instr = f"<s>[INST] {prompt} [/INST]"

# inputs = tokenizer(instr, return_tensors="pt").to(model.device)

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         do_sample=True,
#         temperature=0.2,
#         max_new_tokens=128,
#         eos_token_id=tokenizer.eos_token_id,
#         pad_token_id=tokenizer.pad_token_id,
#     )

# text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(text)
